In [ ]:
!pip install openpyxl boto3 dotenv

In [4]:
from dotenv import load_dotenv
import os
import boto3
import io
import csv
from openpyxl import load_workbook

# Load credentials from .env file
load_dotenv()

os.environ['AWS_ACCESS_KEY_ID'] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = os.getenv('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = os.getenv('AWS_DEFAULT_REGION')

In [ ]:
"""
    Stream an Excel file from S3, convert to CSV row-by-row, and write back to S3.
    Safe for large Excel files. No Spark. No local disk.
"""

# member data
s3 = boto3.client("s3")
# Read Excel from S3 as binary stream
obj = s3.get_object(Bucket="titan-glue-test-data", Key="client_data_excel/1_12082025/member_data/TitanUAE_EnrolmentReport_08122025.xlsx")
excel_stream = io.BytesIO(obj["Body"].read())

# Load workbook in streaming (read-only) mode
wb = load_workbook(
    excel_stream,
    read_only=True,
    data_only=True
)

ws = wb.active

# Stream rows into CSV buffer
csv_buffer = io.StringIO()
writer = csv.writer(csv_buffer)

for row in ws.iter_rows(values_only=True):
    writer.writerow(row)

wb.close()

# Upload CSV back to S3
s3.put_object(
    Bucket="titan-glue-test-data",
    Key="initial_client_data_csv/member_data/member_data.csv",
    Body=csv_buffer.getvalue().encode("utf-8")
)

In [ ]:
# transaction data
s3 = boto3.client("s3")
# Read Excel from S3 as binary stream
obj = s3.get_object(Bucket="titan-glue-test-data", Key="client_data_excel/1_12082025/transaction_data/UAETransactionReportfrominceptiontill_08122025.xlsx")
excel_stream = io.BytesIO(obj["Body"].read())

# Load workbook in streaming (read-only) mode
wb = load_workbook(
    excel_stream,
    read_only=True,
    data_only=True
)

ws = wb.active

# Stream rows into CSV buffer
csv_buffer = io.StringIO()
writer = csv.writer(csv_buffer)

for row in ws.iter_rows(values_only=True):
    writer.writerow(row)

wb.close()

# Upload CSV back to S3
s3.put_object(
    Bucket="titan-glue-test-data",
    Key="initial_client_data_csv/transaction_data/transaction_data.csv",
    Body=csv_buffer.getvalue().encode("utf-8")
)

In [7]:
import boto3, io, csv
from openpyxl import load_workbook

s3 = boto3.client("s3")

# Campaign data
input_bucket = "titan-glue-test-data"
input_key = "client_data_excel/1_12082025/campaign_data/Campaign_Tracker_Updated.xlsx"
output_bucket = "titan-glue-test-data"
output_key = "initial_client_data_csv/campaign_data/campaign_data.csv"

# Read Excel from S3
obj = s3.get_object(Bucket=input_bucket, Key=input_key)
excel_stream = io.BytesIO(obj["Body"].read())

# Load workbook (streaming)
wb = load_workbook(excel_stream, read_only=True, data_only=True)
ws = wb.active

# Prepare CSV
csv_buffer = io.StringIO()
writer = csv.writer(csv_buffer)

rows = ws.iter_rows(values_only=True)

# Handle header explicitly
header = list(next(rows))
header[0] = "MonthYear"          # ⬅️ rename first column
writer.writerow(header)

# Write data rows as-is
for row in rows:
    writer.writerow(row)

wb.close()

# Upload CSV back to S3
s3.put_object(
    Bucket=output_bucket,
    Key=output_key,
    Body=csv_buffer.getvalue().encode("utf-8")
)


{'ResponseMetadata': {'RequestId': 'ZNDZ49B9NA8C3HHC',
  'HostId': 'EBWwTeB085uZ01X2jRRchW0G+9HW1xaROxqJAi31OTGvLOcMDyY7aKaPDhq6894XQDEMWiBZNLQ=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'EBWwTeB085uZ01X2jRRchW0G+9HW1xaROxqJAi31OTGvLOcMDyY7aKaPDhq6894XQDEMWiBZNLQ=',
   'x-amz-request-id': 'ZNDZ49B9NA8C3HHC',
   'date': 'Tue, 27 Jan 2026 08:01:53 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"c50079cafbd9f3585065a35b60e78d25"',
   'x-amz-checksum-crc32': 'MaN/4w==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"c50079cafbd9f3585065a35b60e78d25"',
 'ChecksumCRC32': 'MaN/4w==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}